In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost  import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.preprocessing import scale, StandardScaler
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,precision_score,confusion_matrix,classification_report

In [23]:
tabela = pd.read_csv('../data/02-processed/dados-agua-tratados.csv')

In [24]:
X = tabela.drop("potabilidade", axis = 1).values
y = tabela["potabilidade"].values

In [25]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42)
print("X_train",X_train.shape)
print("X_test",X_test.shape)
print("y_train",y_train.shape)
print("y_test",y_test.shape)

X_train (2293, 9)
X_test (983, 9)
y_train (2293,)
y_test (983,)


In [26]:
# min-max normalization
x_train_max = np.max(X_train)
x_train_min = np.min(X_train)
X_train = (X_train - x_train_min)/(x_train_max-x_train_min)
X_test = (X_test - x_train_min)/(x_train_max-x_train_min)

In [27]:
models = [("RFC",RandomForestClassifier()),
          ("GBM",GradientBoostingClassifier()),
          ("XGB",XGBClassifier()),
          ("LGBM",LGBMClassifier(verbose=-1, force_col_wise=True)),
          ("CATB",CatBoostClassifier(verbose=0))
          ]

In [28]:
finalResult = pd.DataFrame(columns=["Model","Accuracy","Precision"]) # score
cmList = [] # Confusion matrix

for name,model in models:
    if(name == "MLP" or name=="KNN" ):
        # X_train -  X_test scaled
        scaler = StandardScaler()
        scaler.fit(X_train)
        X_train_scaled = scaler.transform(X_train)
        scaler.fit(X_test)
        X_test_scaled = scaler.transform(X_test)

        model.fit(X_train_scaled,y_train) # training
        y_pred = model.predict(X_test_scaled) # prediction
        
        acc = accuracy_score(y_test,y_pred)
        score = precision_score(y_test,y_pred)
        finalResult =pd.concat([finalResult,pd.DataFrame([[name,acc*100,score]],columns=["Model","Accuracy","Precision"])],
                               ignore_index=True)

        cm = confusion_matrix(y_test,y_pred)
        cmList.append((name,cm))
    else:

        model.fit(X_train,y_train) # training
        y_pred = model.predict(X_test) # prediction
        
        acc = accuracy_score(y_test,y_pred)
        score = precision_score(y_test,y_pred)
        finalResult =pd.concat([finalResult,pd.DataFrame([[name,acc*100,score]],columns=["Model","Accuracy","Precision"])],
                               ignore_index=True)

        cm = confusion_matrix(y_test,y_pred)
        cmList.append((name,cm))

c:\Users\Robert Fernandes\Documents\Workspace\analise-agua\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [29]:
finalResult

,Model,Accuracy,Precision
0,RFC,79.043744,0.78777
1,GBM,76.500509,0.77551
2,XGB,79.348932,0.755486
3,LGBM,79.959308,0.76489
4,CATB,80.671414,0.821168


In [30]:
model_params = {
    "CatBoost":{
        "model":CatBoostClassifier(verbose=0),
        "params":{
            "iterations":[200,500,1000],
            "learning_rate":[0.01,0.03,0.1],
            "depth":[4,5,8]
        }
    },
    "Random Forest":{
        "model":RandomForestClassifier(),
        "params":{
            "n_estimators":[50,100,200,500],
            "max_features":["sqrt","log2"],
            "max_depth":list(range(1,21,3)),
            "min_samples_split":[5,10,20] 
        }
    },
    "XGB":{
        "model":XGBClassifier(verbose=0),
        "params":{
            "n_estimators":[100,500,1000,2000],
            "subsample":[0.6,0.8,1],
            "max_depth":[3,5,6,7],
            "learning_rate":[0.1,0.001,0.01]
        }
    },
    "LGBM":{
        "model":LGBMClassifier(verbose=-1),
        "params":{
            "learning_rate":[0.001,0.01,0.1],
            "n_estimators":[200,500,100],
            "max_depth":[1,2,3,5,8]},
        },
    }

In [31]:
cv = RepeatedStratifiedKFold(n_splits=5,n_repeats=2)
scores = []

for model_name,params in model_params.items():
    rs = RandomizedSearchCV(params["model"],params["params"],cv=cv,n_iter=10) 
    rs.fit(X,y)
    scores.append([model_name,dict(rs.best_params_),rs.best_score_])

c:\Users\Robert Fernandes\Documents\Workspace\analise-agua\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [12:12:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Robert Fernandes\Documents\Workspace\analise-agua\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [12:12:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Robert Fernandes\Documents\Workspace\analise-agua\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [12:12:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Robert Fernandes\Documents\Workspace\analise-agua\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [12:12:27] WARNING

In [32]:
scores

[['CatBoost',
  {'learning_rate': 0.01, 'iterations': 1000, 'depth': 8},
  np.float64(0.8040278812139265)],
 ['Random Forest',
  {'n_estimators': 500,
   'min_samples_split': 20,
   'max_features': 'log2',
   'max_depth': 10},
  np.float64(0.7991468069260846)],
 ['XGB',
  {'subsample': 0.8,
   'n_estimators': 2000,
   'max_depth': 7,
   'learning_rate': 0.01},
  np.float64(0.7953304784956247)],
 ['LGBM',
  {'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.01},
  np.float64(0.7918185626512753)]]

In [33]:
# Modelos
modelos = {
    "CatBoost": CatBoostClassifier(
        learning_rate=0.01,
        verbose=0,
        iterations=1000,
        depth=8
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        min_samples_split=20,
        max_features="sqrt",
        max_depth=16,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        subsample=0.8,
        n_estimators=500,
        max_depth=5,
        learning_rate=0.1,
        verbosity=0,
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.01,
        verbosity=-1,
        random_state=42
    )
}

In [34]:
for nome, modelo in modelos.items():

    # treino
    modelo.fit(X_train, y_train)

    # previsão
    y_pred = modelo.predict(X_test)

    # métricas
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)

    print(f"\nModelo: {nome}")
    print(f"Acurácia: {acc:.4f}")
    print(f"Precisão: {prec:.4f}")


Modelo: CatBoost
Acurácia: 0.8067
Precisão: 0.8259

Modelo: Random Forest
Acurácia: 0.7945
Precisão: 0.7993

Modelo: XGBoost
Acurácia: 0.7904
Precisão: 0.7516

Modelo: LightGBM
Acurácia: 0.7915
Precisão: 0.8182


c:\Users\Robert Fernandes\Documents\Workspace\analise-agua\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
